# P2 — Vanilla DistilBERT Baseline
## Notebook 05: `05_vanilla_baseline.ipynb`

**Project:** Optimizing Inference Latency in Enterprise NLP via Task-Specific Knowledge Distillation  
**Group 15 | Section 2241044 | ITER, Siksha 'O' Anusandhan University**

---

### Purpose of This Notebook

This notebook trains a **controlled baseline model** — DistilBERT fine-tuned directly on gold labels with no pseudo-labels and no Knowledge Distillation.

This answers the core research question:

> *Did KD actually help over plain supervised fine-tuning?*

All hyperparameters (learning rate, epochs, batch size, class weights, dropout) are **identical** to the KD pipeline in `03_distilbert_finetuning.ipynb` to ensure a fair comparison.

| | KD Pipeline | Vanilla Baseline (this notebook) |
|---|---|---|
| Training labels | Llama pseudo-labels | Gold human labels |
| Architecture | Dual-head (sentiment + urgency) | Single-head (sentiment only) |
| Annotation cost | Zero | High |
| Hyperparameters | 2e-5, 5 epochs, batch=16 | **Identical** |

---

### Cell 1 — Restore Session

Reinstalls all required libraries, mounts Google Drive, loads the train/val/test splits saved during Step 1, and applies the same label mapping used throughout the pipeline.

**Run this first every time you open a new Colab session.**

In [1]:
!pip install transformers scikit-learn pandas numpy torch -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import torch

# Load gold-labelled splits
train_df = pd.read_csv('/content/drive/MyDrive/KD_Project/train.csv')
val_df   = pd.read_csv('/content/drive/MyDrive/KD_Project/val.csv')
test_df  = pd.read_csv('/content/drive/MyDrive/KD_Project/test.csv')

# Label maps — identical to KD pipeline
SENTIMENT_MAP = {"negative": 0, "neutral": 1, "positive": 2}
INV_SENTIMENT = {v: k for k, v in SENTIMENT_MAP.items()}

train_df["sentiment_id"] = train_df["sentiment"].map(SENTIMENT_MAP)
val_df["sentiment_id"]   = val_df["sentiment"].map(SENTIMENT_MAP)
test_df["sentiment_id"]  = test_df["sentiment"].map(SENTIMENT_MAP)

print(f"Train : {len(train_df)} samples")
print(f"Val   : {len(val_df)} samples")
print(f"Test  : {len(test_df)} samples")
print(f"GPU   : {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
Train : 3876 samples
Val   : 485 samples
Test  : 485 samples
GPU   : Tesla T4


### Cell 2 — Build Dataset and DataLoaders

Defines a standard PyTorch `Dataset` class that tokenises financial sentences using the DistilBERT tokenizer (`max_length=128`).

Creates three DataLoaders:
- `train_loader` — batch size 16, shuffled
- `val_loader` — batch size 32, sequential
- `test_loader` — batch size 32, sequential

These are identical in structure to the KD pipeline DataLoaders, ensuring the baseline comparison is fair.

In [2]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

class FinancialDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            list(texts),
            truncation     = True,
            padding        = True,
            max_length     = max_length,
            return_tensors = 'pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids'     : self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label'         : self.labels[idx]
        }

train_dataset = FinancialDataset(
    train_df["text"].values,
    train_df["sentiment_id"].values,
    tokenizer
)
val_dataset = FinancialDataset(
    val_df["text"].values,
    val_df["sentiment_id"].values,
    tokenizer
)
test_dataset = FinancialDataset(
    test_df["text"].values,
    test_df["sentiment_id"].values,
    tokenizer
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")
print(f"Dataset ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train batches : 243
Val batches   : 16
Test batches  : 16
Dataset ready.


### Cell 3 — Define Vanilla Single-Head Model

Defines `VanillaDistilBERT` — a standard DistilBERT with a **single** classification head for sentiment only.

Key difference from the KD model:
- **KD model:** Dual-head (sentiment head + urgency head)
- **Vanilla model:** Single-head (sentiment head only)

Both share the same DistilBERT-base-uncased encoder (66M parameters), same dropout rate (0.3), and same hidden size (768). This isolates the effect of KD training versus standard supervised fine-tuning.

In [3]:
from transformers import DistilBertModel

class VanillaDistilBERT(nn.Module):
    """
    Standard single-head DistilBERT for sentiment classification.
    No urgency head. No KD. Trained on gold labels only.
    This is the controlled baseline for comparison against the KD pipeline.
    """
    def __init__(self, num_classes=3, dropout=0.3):
        super(VanillaDistilBERT, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs    = self.distilbert(
            input_ids      = input_ids,
            attention_mask = attention_mask
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        return self.classifier(cls_output)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = VanillaDistilBERT().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Device           : {device}")
print(f"Total parameters : {total_params:,}")
print(f"Model ready.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device           : cuda
Total parameters : 66,365,187
Model ready.


### Cell 4 — Class Weights and Training Setup

Computes class weights using `sklearn` balanced weighting — identical to the KD pipeline to ensure a controlled comparison.

Training hyperparameters are kept identical to `03_distilbert_finetuning.ipynb`:
- Learning rate: `2e-5`
- Epochs: `5`
- Batch size: `16`
- Weight decay: `0.01`
- Warmup: 10% of total steps
- Scheduler: Linear warmup + decay

The only intentional difference is the training data source: gold labels here vs pseudo-labels in the KD pipeline.

In [4]:
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup

# Same class weights as KD pipeline — controlled comparison
sentiment_weights = compute_class_weight(
    class_weight = 'balanced',
    classes      = np.array([0, 1, 2]),
    y            = train_df["sentiment_id"].values
)
weights_tensor = torch.tensor(sentiment_weights, dtype=torch.float).to(device)

print("Class weights:")
for i, w in enumerate(sentiment_weights):
    print(f"  {INV_SENTIMENT[i]:>10} : {w:.4f}")

# Loss, optimiser, scheduler — identical hyperparameters to KD run
criterion   = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer   = torch.optim.AdamW(
    model.parameters(),
    lr           = 2e-5,
    weight_decay = 0.01
)

EPOCHS       = 5
TOTAL_STEPS  = len(train_loader) * EPOCHS
WARMUP_STEPS = int(0.1 * TOTAL_STEPS)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = WARMUP_STEPS,
    num_training_steps = TOTAL_STEPS
)

print(f"\nEpochs       : {EPOCHS}")
print(f"Total steps  : {TOTAL_STEPS}")
print(f"Warmup steps : {WARMUP_STEPS}")
print(f"Training setup complete.")

Class weights:
    negative : 2.6749
     neutral : 0.5610
    positive : 1.1853

Epochs       : 5
Total steps  : 1215
Warmup steps : 121
Training setup complete.


### Cell 5 — Training Loop

Standard training loop with per-epoch validation. Identical structure to the KD training loop in `03_distilbert_finetuning.ipynb`.

At each epoch:
1. Forward pass + weighted cross-entropy loss
2. Backpropagation with gradient clipping (max_norm=1.0)
3. Validation pass — compute Macro F1 on gold val labels
4. Save best model state based on Val Macro F1

Expected runtime: **8–10 minutes** on T4 GPU.

In [5]:
from sklearn.metrics import f1_score
import time

best_val_f1      = 0.0
best_model_state = None
train_history    = []

print("Starting vanilla baseline training...\n")
print(f"{'Epoch':<8}{'Train Loss':<14}{'Val Loss':<12}{'Val F1 (Macro)':<18}{'Time'}")
print("-" * 60)

for epoch in range(EPOCHS):
    epoch_start = time.time()

    # ── Training ──────────────────────────────────────────────
    model.train()
    total_train_loss = 0

    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # ── Validation ────────────────────────────────────────────
    model.eval()
    total_val_loss = 0
    all_preds      = []
    all_labels     = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            logits   = model(input_ids, attention_mask)
            val_loss = criterion(logits, labels)

            total_val_loss += val_loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)
    val_f1       = f1_score(all_labels, all_preds, average='macro')
    epoch_time   = time.time() - epoch_start

    print(f"Epoch {epoch+1:<4} {avg_train_loss:<14.4f}{avg_val_loss:<12.4f}{val_f1:<18.4f}{epoch_time:.1f}s")

    train_history.append({
        "epoch"      : epoch + 1,
        "train_loss" : avg_train_loss,
        "val_loss"   : avg_val_loss,
        "val_f1"     : val_f1
    })

    if val_f1 > best_val_f1:
        best_val_f1      = val_f1
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"         -> New best model saved (Val F1: {val_f1:.4f})")

print(f"\nTraining complete. Best Val Macro F1 : {best_val_f1:.4f}")

Starting vanilla baseline training...

Epoch   Train Loss    Val Loss    Val F1 (Macro)    Time
------------------------------------------------------------
Epoch 1    0.7645        0.3816      0.7792            40.7s
         -> New best model saved (Val F1: 0.7792)
Epoch 2    0.3334        0.3170      0.8381            41.6s
         -> New best model saved (Val F1: 0.8381)
Epoch 3    0.1843        0.3496      0.8405            43.8s
         -> New best model saved (Val F1: 0.8405)
Epoch 4    0.1042        0.4488      0.8411            43.1s
         -> New best model saved (Val F1: 0.8411)
Epoch 5    0.0578        0.4633      0.8363            43.5s

Training complete. Best Val Macro F1 : 0.8411


### Cell 6 — Test Set Evaluation

Loads the best saved model weights and runs inference on the held-out test set (485 samples).

Reports:
- Per-class Precision, Recall, F1
- Overall Macro F1 and Accuracy

This is the primary number for comparison against the KD pipeline result of **Macro F1 = 0.7452**.

In [6]:
from sklearn.metrics import classification_report

model.load_state_dict(best_model_state)
model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        logits = model(input_ids, attention_mask)
        preds  = torch.argmax(logits, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy())

macro_f1 = f1_score(all_labels, all_preds, average='macro')
accuracy  = (np.array(all_preds) == np.array(all_labels)).mean()

print("=" * 55)
print("VANILLA BASELINE — TEST SET RESULTS")
print("=" * 55)
print(classification_report(
    all_labels, all_preds,
    target_names=["negative", "neutral", "positive"]
))
print(f"Macro F1 Score : {macro_f1:.4f}")
print(f"Accuracy       : {accuracy:.4f}")

VANILLA BASELINE — TEST SET RESULTS
              precision    recall  f1-score   support

    negative       0.77      0.87      0.82        61
     neutral       0.88      0.85      0.86       288
    positive       0.76      0.78      0.77       136

    accuracy                           0.83       485
   macro avg       0.80      0.83      0.82       485
weighted avg       0.83      0.83      0.83       485

Macro F1 Score : 0.8167
Accuracy       : 0.8309


### Cell 7 — Direct Comparison Table

Side-by-side comparison of the KD pipeline vs vanilla baseline across all key metrics.

This is the **central research finding** of the project. The gap between these two models tells us:

| Gap Size | Research Interpretation |
|---|---|
| ≤ 0.05 | KD matches gold-label fine-tuning — zero-annotation KD is viable |
| 0.05–0.10 | Small accuracy cost offset by zero annotation requirement |
| > 0.10 | Teacher quality ceiling is the binding constraint |

The comparison table and finding statement are formatted for direct inclusion in the paper.

In [7]:
# KD pipeline results from Step 3 and Step 4
kd_results = {
    "negative_f1" : 0.75,
    "neutral_f1"  : 0.83,
    "positive_f1" : 0.66,
    "macro_f1"    : 0.7452,
    "accuracy"    : 0.7773,
    "annotation"  : "Zero (pseudo-labels)",
    "params"      : "66M"
}

# Vanilla baseline results computed above
per_class_f1 = f1_score(all_labels, all_preds, average=None)
vanilla_results = {
    "negative_f1" : round(per_class_f1[0], 4),
    "neutral_f1"  : round(per_class_f1[1], 4),
    "positive_f1" : round(per_class_f1[2], 4),
    "macro_f1"    : round(macro_f1, 4),
    "accuracy"    : round(accuracy, 4),
    "annotation"  : "High (gold labels)",
    "params"      : "66M"
}

print("=" * 70)
print("KD PIPELINE vs VANILLA BASELINE — DIRECT COMPARISON")
print("=" * 70)
print(f"{'Metric':<25} {'KD Pipeline':<22} {'Vanilla Baseline':<22}")
print("-" * 70)
print(f"{'Negative F1':<25} {kd_results['negative_f1']:<22} {vanilla_results['negative_f1']:<22}")
print(f"{'Neutral F1':<25} {kd_results['neutral_f1']:<22} {vanilla_results['neutral_f1']:<22}")
print(f"{'Positive F1':<25} {kd_results['positive_f1']:<22} {vanilla_results['positive_f1']:<22}")
print(f"{'Macro F1':<25} {kd_results['macro_f1']:<22} {vanilla_results['macro_f1']:<22}")
print(f"{'Accuracy':<25} {kd_results['accuracy']:<22} {vanilla_results['accuracy']:<22}")
print(f"{'Annotation Cost':<25} {kd_results['annotation']:<22} {vanilla_results['annotation']:<22}")
print(f"{'Parameters':<25} {kd_results['params']:<22} {vanilla_results['params']:<22}")
print("=" * 70)

# Gap analysis and finding statement
gap = vanilla_results['macro_f1'] - kd_results['macro_f1']
print(f"\nMacro F1 gap (vanilla - KD) : {gap:+.4f}")

if abs(gap) <= 0.05:
    print("\nFINDING: KD achieves comparable accuracy to gold-label fine-tuning")
    print("         with zero annotation cost — core contribution validated.")
elif gap > 0.05:
    print(f"\nFINDING: Vanilla outperforms KD by {gap:.4f}.")
    print("         Teacher quality ceiling is the limiting factor.")
    print("         Documents real limitation of black-box KD with small LLM teachers.")
else:
    print(f"\nFINDING: KD outperforms vanilla by {abs(gap):.4f}.")
    print("         Pseudo-labels provide richer training signal than gold labels alone.")

# Save comparison to Drive
comparison_df = pd.DataFrame([
    {"model": "KD Pipeline",      **kd_results},
    {"model": "Vanilla Baseline", **vanilla_results}
])
comparison_df.to_csv(
    '/content/drive/MyDrive/KD_Project/baseline_comparison.csv',
    index=False
)
print(f"\nComparison table saved to Drive.")

KD PIPELINE vs VANILLA BASELINE — DIRECT COMPARISON
Metric                    KD Pipeline            Vanilla Baseline      
----------------------------------------------------------------------
Negative F1               0.75                   0.8154                
Neutral F1                0.83                   0.8637                
Positive F1               0.66                   0.7709                
Macro F1                  0.7452                 0.8167                
Accuracy                  0.7773                 0.8309                
Annotation Cost           Zero (pseudo-labels)   High (gold labels)    
Parameters                66M                    66M                   

Macro F1 gap (vanilla - KD) : +0.0715

FINDING: Vanilla outperforms KD by 0.0715.
         Teacher quality ceiling is the limiting factor.
         Documents real limitation of black-box KD with small LLM teachers.

Comparison table saved to Drive.


### Cell 8 — Save Baseline Model and Training History

Saves the best vanilla baseline model weights, tokenizer, and epoch-wise training history to Google Drive.

Saved files:
- `distilbert_vanilla/model_weights.pt` — best model checkpoint
- `distilbert_vanilla/tokenizer_*` — tokenizer files
- `distilbert_vanilla/training_history.csv` — loss and F1 per epoch

These are committed to GitHub under `results/` for reproducibility.

In [8]:
import os

BASELINE_DIR = '/content/drive/MyDrive/KD_Project/distilbert_vanilla'
os.makedirs(BASELINE_DIR, exist_ok=True)

torch.save(best_model_state, f'{BASELINE_DIR}/model_weights.pt')
tokenizer.save_pretrained(BASELINE_DIR)
pd.DataFrame(train_history).to_csv(
    f'{BASELINE_DIR}/training_history.csv',
    index=False
)

print(f"Baseline model saved.")
print(f"\nSaved files:")
print(f"  {BASELINE_DIR}/model_weights.pt")
print(f"  {BASELINE_DIR}/tokenizer files")
print(f"  {BASELINE_DIR}/training_history.csv")
print(f"\nNext step: commit baseline_comparison.csv to GitHub results/ folder.")

Baseline model saved.

Saved files:
  /content/drive/MyDrive/KD_Project/distilbert_vanilla/model_weights.pt
  /content/drive/MyDrive/KD_Project/distilbert_vanilla/tokenizer files
  /content/drive/MyDrive/KD_Project/distilbert_vanilla/training_history.csv

Next step: commit baseline_comparison.csv to GitHub results/ folder.
